In [1]:
import pandas as pd

## Load data

In [2]:
df_meta_train = pd.read_csv("../../data/monte_outputs/pancancer/pancancer_meta_with_predictions.csv")
df_meta_test = pd.read_csv("../../data/monte_outputs/pancancer/pancancer_meta_test_with_predictions.csv")

## Calculate the correlation between different metrics and predicted ESTIMATE

In [ ]:
def calculate_correlations_by_cancer(df):

    pred_metric = "pred_ESTIMATE"
    metrics = ["ABSOLUTE", "ESTIMATE", "LUMP", "CPE"]
    cancer_types = df_meta_train["Cancer.type"].unique()

    result = pd.DataFrame(index=cancer_types, columns=metrics + ["n_samples"], dtype=float)

    for cancer_type in cancer_types:
        df_cancer = df[df["Cancer.type"] == cancer_type]
        result.loc[cancer_type, "n_samples"] = len(df_cancer)

        for metric in metrics:
            # use only valid paired values for correlation
            pair = df_cancer[[pred_metric, metric]].dropna()

            if len(pair) < 2:
                corr = float("nan")
            elif pair[pred_metric].std() == 0 or pair[metric].std() == 0:
                # No variance in one or both columns
                corr = float("nan")
            else:
                corr = pair[pred_metric].corr(pair[metric])

            result.loc[cancer_type, metric] = corr

    return result

In [20]:
df_corr_train = calculate_correlations_by_cancer(df_meta_train)
df_corr_test = calculate_correlations_by_cancer(df_meta_test)

/grain/wl61/github/ylab/MONTE-analysis/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/grain/wl61/github/ylab/MONTE-analysis/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/grain/wl61/github/ylab/MONTE-analysis/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/grain/wl61/github/ylab/MONTE-analysis/.venv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Note: Owing to OV samples in test set has only 3 samples, our predictions are all "1". Therefore, there is no variance to calculate the correlation.

In [24]:
df_meta_test_ov = df_meta_test[df_meta_test["Cancer.type"] == "OV"]
df_meta_test_ov["pred_ESTIMATE"]

731     1.0
997     1.0
1864    1.0
Name: pred_ESTIMATE, dtype: float64

## Save results

In [22]:
df_corr_train.to_csv("../../data/monte_outputs/pancancer/pancancer_correlation_results_train.csv", index=True)
df_corr_test.to_csv("../../data/monte_outputs/pancancer/pancancer_correlation_results_test.csv", index=True)